<a href="https://colab.research.google.com/github/metarun/Mechanistic-Interpretability-Exploration/blob/main/Attention_Mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Understanding Simple Self-Attention (Without Trainable Weights)**

In this notebook, I'll implement a simplified version of the Self-Attention mechanism. For educational purposes, I'll assume the **Query (Q)**, **Key (K)**, and **Value (V)** matrices are derived directly from the input embeddings without initial linear transformations.

### The Mathematical Foundation
The core formula for scaled dot-product attention is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In a full Transformer layer, these are calculated as:

$$\text{head}_i = \text{Attention}\left(QW_i^Q,\; KW_i^K,\; VW_i^V\right)$$

### **The Self-Attention Mechanism: Step-by-Step**

The self-attention formula is defined as:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

#### **Key Components:**

1.  **Query (Q), Key (K), and Value (V)**:
    *   **Query**: What we are looking for (the 'search term').
    *   **Key**: What each token offers (the 'index' or 'label').
    *   **Value**: The actual information associated with the token.

2.  **Attention Scores ($QK^T$)**: Measures similarity between the Query and all Keys.

3.  **Scaling ($\sqrt{d_k}$)**: Prevents gradients from vanishing by keeping dot-product values manageable.

4.  **Softmax**: Normalizes scores into weights that sum to 1, creating a probability distribution over the sequence.

5.  **Weighted Sum**: Multiplies the weights by the Value vectors to create a context-aware representation.

# **Step by Step Calcluation of Attention Just for 1 Token for now**

## **Data Setup: Defining Our Input Sentance / Tokens**

In [42]:
import pandas as pd

# Define the labels for the input tokens based on the comment in the 'inputs' tensor --> Below
token_labels = [
    "Your",      # x^1
    "journey",   # x^2
    "starts",    # x^3
    "with",      # x^4
    "one",       # x^5
    "step"       # x^6
]
token_labels

['Your', 'journey', 'starts', 'with', 'one', 'step']

## **Data Setup: Initlislising Our Input Tokens with random weights**


In [57]:
import torch

# Define input embeddings for our sample sentence:
# "Your journey starts with one step"
# Each row represents a token, each column represents a dimension (d_model = 3)
inputs = torch.tensor(
     [[0.43, 0.15, 0.89], # 'Your'    (x^1)
      [0.55, 0.87, 0.66], # 'journey' (x^2)
      [0.57, 0.85, 0.64], # 'starts'  (x^3)
      [0.22, 0.58, 0.33], # 'with'    (x^4)
      [0.77, 0.25, 0.10], # 'one'     (x^5)
      [0.05, 0.80, 0.55], # 'step'    (x^6)
    ]
)

print(f"Input shape: {inputs.shape} (6 tokens, 3 dimensions)")

Input shape: torch.Size([6, 3]) (6 tokens, 3 dimensions)


## **Data Setup: Creating Q K V metrices for our calculations for Attention**


In [58]:
K = inputs
Q = inputs
V = inputs

## **Step 1: Calculating the Dot Product Similarity ($QK^T$)**

> Add blockquote



I first compute the similarity between tokens. I'll start by looking at a single query (the token 'journey') against all other tokens (keys). This dot product serves as a measure of alignment: the higher the score, the more relevant the key is to the query.

In [45]:
# I select the 2nd token ('journey') as my Query (Q)
query_journey = inputs[1]

# For this simplified version, I'll use the transpose of the input matrix as my Keys (K)
# This allows me to perform a dot product between the query and every token in the sequence simultaneously
Kt = K.T

# Calculate attention scores for the 'journey' token
# Score(q, k) = q · k
attention_score_journey = query_journey @ Kt

print("Attention scores for 'journey':", attention_score_journey)

Attention scores for 'journey': tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


## **Step 2: Scaling and Normalization**

For this simple example, I assume $d_k=1$ (no scaling). I use the Softmax function to turn the raw scores into weights. This step is crucial because it ensures all weights are positive and sum to 1, effectively creating a probability distribution that dictates how much 'attention' I pay to each token.

In [46]:
# Step 2: Apply Softmax to obtain Attention Weights
# These weights determine how much focus the model puts on other parts of the sentence.

def softmax_naive(x):
    """A basic implementation of the softmax function."""
    return torch.exp(x) / torch.exp(x).sum(dim=0)

# Comparing manual vs. PyTorch implementation

# Applying the manual softmax function to our specific query scores
manual_weights_journey = softmax_naive(attention_score_journey)
print(f"manual_weights_journey -- > {manual_weights_journey}")


# Using PyTorch's optimized softmax for the 'journey' token weights
pytorch_weights_journey = torch.softmax(attention_score_journey, dim=0)
print(f"pytorch_weights_journey -- > {pytorch_weights_journey}")

manual_weights_journey -- > tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
pytorch_weights_journey -- > tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


## **Step 3: Compute the Context Vector**
I multiply the attention weights by the input embeddings to get a single vector

That represents 'journey' in the context of the entire sentence.

**Find context vector / Atention with respect to 2nd input token - journey $$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$**

In [47]:
context_vector_journey = pytorch_weights_journey @ V

print("Original 'journey' vector:  ", inputs[1])
print("Context-aware representation:", context_vector_journey)

Original 'journey' vector:   tensor([0.5500, 0.8700, 0.6600])
Context-aware representation: tensor([0.4419, 0.6515, 0.5683])


So our token - Journey was **[0.55, 0.87, 0.66]** but now we have added attention weight and also gave it the context for entire sentance hence now it is transformed and is now more alingend with context of entire sentance hence now it is **[0.4419, 0.6515, 0.5683]**

# Scaling to the Entire Sequence

Instead of calculating one token at a time, I can compute the context vectors for the entire sentence simultaneously using matrix multiplication. This is where the efficiency of Transformers truly shines, as I can process the whole sequence in parallel.

* input_query == > inputs
* Key == > inputs ( For this example)
* Value = > inputs ( For this example)

**Lets calculate ($K^T$)**

In [59]:
Kt = K.T
Kt

tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])

### **Lets calculate ($QK^T$)**

In [49]:
QKt = Q @ Kt
QKt

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [50]:
# Show Attention Score for all inputs with respect to other tokens
QKt_df = pd.DataFrame(QKt.numpy(), index=token_labels, columns=token_labels)

print("Attention Score (rows are Queries, columns are dimensions of the context vector):")
display(QKt_df)

Attention Score (rows are Queries, columns are dimensions of the context vector):


,Your,journey,starts,with,one,step
Your,0.9995,0.9544,0.9422,0.4753,0.4576,0.6310
journey,0.9544,1.4950,1.4754,0.8434,0.7070,1.0865
starts,0.9422,1.4754,1.4570,0.8296,0.7154,1.0605
with,0.4753,0.8434,0.8296,0.4937,0.3474,0.6565
one,0.4576,0.7070,0.7154,0.3474,0.6654,0.2935
step,0.6310,1.0865,1.0605,0.6565,0.2935,0.9450


## **Lets take Softmax of ($QK^T$)**

In [51]:
SoftMaxQKt = torch.softmax(QKt, dim=1)
SoftMaxQKt

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [52]:
# Show Attention Score for all inputs with respect to other tokens
SoftMaxQKt_df = pd.DataFrame(SoftMaxQKt.numpy(), index=token_labels, columns=token_labels)

print("Attention Score (rows are Queries, columns are dimensions of the context vector):")
display(SoftMaxQKt_df)

Attention Score (rows are Queries, columns are dimensions of the context vector):


,Your,journey,starts,with,one,step
Your,0.209835,0.200581,0.198149,0.124228,0.122049,0.145158
journey,0.138548,0.237891,0.233274,0.123992,0.108182,0.158114
starts,0.139008,0.236921,0.232602,0.124204,0.110800,0.156464
with,0.143527,0.207394,0.204552,0.146192,0.126295,0.172039
one,0.152611,0.195839,0.197491,0.136687,0.187859,0.129514
step,0.138471,0.218364,0.212759,0.142048,0.098806,0.189552


## **Lets find the attention/context vectors for all tokens**

**$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$**

In [53]:
# Multiple SoftMaxQKt with V ==> inputs
AttentionWeights = SoftMaxQKt @ V
AttentionWeights

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [54]:
# Create the DataFrame with correct dimensions
attention_df = pd.DataFrame(AttentionWeights.numpy(), index=token_labels)

print("Attention Weights (rows are Queries, columns are dimensions of the context vector):")
display(attention_df)

Attention Weights (rows are Queries, columns are dimensions of the context vector):


,0,1,2
Your,0.442059,0.593099,0.578989
journey,0.441866,0.651482,0.568309
starts,0.443128,0.649595,0.567073
with,0.430390,0.629828,0.551027
one,0.467102,0.590993,0.526597
step,0.417724,0.650323,0.564535


This DataFrame `attention_df` now shows the new, context-aware representations for each of your original tokens. For instance, the first row, labeled 'Your', now has a vector `[0.4421, 0.5931, 0.5790]`, which is its updated representation incorporating information from all other tokens in the sentence through the self-attention mechanism.

# Single step do all these

In [55]:
# Compute Attention in a single efficient operation
# 1. inputs @ inputs.T -> Compute all dot product scores (Q * K^T)
# 2. torch.softmax(..., dim=1) -> Normalize scores across rows
# 3. @ inputs -> Multiply weights by values (V)

attention_outputs = torch.softmax(inputs @ inputs.T, dim=1) @ inputs

print("Full Attention Output Matrix:")
display(attention_outputs)

Full Attention Output Matrix:


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [56]:
# Final Step: Visualizing the results in a readable format
# I create a DataFrame to map the context-aware vectors back to their original tokens.
# This final representation is what a Transformer block would pass to the next layer.
final_results_df = pd.DataFrame(attention_outputs.numpy(), index=token_labels)

print("Context-Aware Token Representations (Output of Self-Attention):")
display(final_results_df)

Context-Aware Token Representations (Output of Self-Attention):


,0,1,2
Your,0.442059,0.593099,0.578989
journey,0.441866,0.651482,0.568309
starts,0.443128,0.649595,0.567073
with,0.430390,0.629828,0.551027
one,0.467102,0.590993,0.526597
step,0.417724,0.650323,0.564535


## **Summary**
By applying self-attention, I have transformed static embeddings into dynamic, context-aware representations. Each token's new vector now contains rich information about its relationship with every other token in the sequence, allowing the model to understand the nuances of the language.